# Fashion-MNIST Submission Notebook

Training notebook:
https://colab.research.google.com/drive/1r6s5XDZGa_L8_YHZZ0MZgjth9-kxHdvX?usp=drive_link

## How I trained this model
- I used the Fashion-MNIST training CSV and normalized pixel values to `[0, 1]`.
- I trained a grayscale ResNet-style base model on all 10 classes.
- I trained multiple seeds and averaged their probabilities for a more stable ensemble prediction.
- I added a specialist model focused on the confusing classes (especially Shirt-related confusion).
- I fused base + specialist probabilities to improve the hardest class decisions.
- I exported the final test probabilities to `test_probs_resnet_specialist.npy`.
- This notebook converts that `.npy` into `submission.csv` for competition upload.

In [ ]:
import numpy as np
import pandas as pd
import os

# ── Configure these two paths ────────────────────────────────────────────────
# Path to your saved probability array (.npy).
# On Kaggle: upload the file as a dataset input; it will appear under /kaggle/input/
NPY_PATH = '/kaggle/input/fashion-mnist-probs/test_probs_resnet_specialist.npy'

# Path to the sample submission provided by the competition.
SAMPLE_SUB_PATH = '/kaggle/input/ai-biz-2026-spring-task-3/fashion-mnist_test_sample_submission.csv'
# ─────────────────────────────────────────────────────────────────────────────

assert os.path.exists(NPY_PATH), f'npy not found: {NPY_PATH}'
assert os.path.exists(SAMPLE_SUB_PATH), f'sample submission not found: {SAMPLE_SUB_PATH}'

test_probs = np.load(NPY_PATH)
print('Loaded probabilities shape:', test_probs.shape)
print('Min prob:', test_probs.min(), '| Max prob:', test_probs.max())
print('Rows sum to 1 (sample check):', np.allclose(test_probs.sum(axis=1), 1.0, atol=1e-4))

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
print('Sample submission shape:', sample_sub.shape)
print(sample_sub.head())

assert len(test_probs) == len(sample_sub), (
    f'Row count mismatch: npy has {len(test_probs)} rows, sample sub has {len(sample_sub)}'
)

In [ ]:
test_pred = np.argmax(test_probs, axis=1)

submission = sample_sub.copy()
submission['label'] = test_pred

out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Label distribution:')
print(submission['label'].value_counts().sort_index())
submission.head(10)